# Stage6:主体驱动任务 · 无 LoRA 基座 —— 两种注意力机制对比(n_cond=1)

**与 stage5 的关系**:同一个问题(默认模式 vs 独立条件模式对基座的原生影响),
换成**主体驱动**的条件摆位 —— 这正好去掉了 stage5 里两个干扰因素:

| | stage5(空间对齐) | stage6(主体驱动,本实验) |
|---|---|---|
| 条件图 | canny 边缘图(二值、有噪声) | **原始 RGB 照片**(自然、稠密) |
| 摆位 | `position_scale`,与主图坐标**重叠** | `position_delta`,与主图**错开并列** |
| RoPE 交互 | 零相对位置,注意力天然偏高 | 相对位置一个图宽,无重叠放大 |
| LoRA | 无 | 无(同左,adapter=None) |

6 张条件图全部来自仓库 assets,其中 5 张的 prompt 取自官方 `examples/subject*.ipynb`
原文;`book.jpg` 官方只用于 inpainting,prompt 为按官方 "this item" 句式自拟。

**用法**:只改 cell 1 的 `TARGET_SIZE`(256 / 512 / 1024),条件图尺寸和 `position_delta`
自动跟随(主体驱动惯例:条件尺寸 = 目标尺寸,delta = 一个图宽的 latent 格数)。
输出目录 `repro/subject_base_compare_<TARGET_SIZE>/`,产物 `comparison_grid.png` 同 stage4/5。

**读图口径**:裸基座不会"迁移主体"(那是 subject LoRA 学的),两列都不还原主体是正常的。
看的是条件 token 并列注入后,哪种交互方式对主图的干扰更大(伪影/色偏/构图崩坏),
以及与 stage5 对比:换成干净条件 + 错开摆位后,两种模式的差距还剩多少。

In [1]:
import os
os.chdir("..")  # 仓库根目录(与其它 repro notebook 一致)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import sys, json, time, random
sys.path.insert(0, "."); sys.path.insert(0, "repro")

import torch
# 与 stage3/probe 相同的兜底:torch 2.8 + diffusers 0.38 的 VAE conv_in 问题
torch.backends.cudnn.enabled = False

from PIL import Image
from kvcache_benchmark import load_pipeline
from omini.pipeline.flux_omini import Condition, generate

# ── 唯一需要改的开关:生成分辨率 ──────────────────────────────────────
TARGET_SIZE = 512                       # 256 / 512 / 1024,其余自动跟随
# ──────────────────────────────────────────────────────────────────
COND_SIZE   = TARGET_SIZE               # 主体驱动惯例:条件尺寸 = 目标尺寸(无压缩)
# WHY delta = TARGET//16:latent 网格是像素/16,平移一个图宽 → 条件与主图并列不重叠,
# 与官方 subject_512 的 (0, 32) 约定一致(512//16=32);方向任意,保持一致即可
POS_DELTA   = (0, TARGET_SIZE // 16)
STEPS       = 28                        # dev 质量档
SEEDS       = [42, 1234]
OUT_DIR     = f"repro/subject_base_compare_{TARGET_SIZE}"  # 随 TARGET_SIZE 切换,多档并存
os.makedirs(OUT_DIR, exist_ok=True)
os.environ["OUT_DIR"] = OUT_DIR   # 同步给可视化 cell 的 shell magic 用
print(f"[DEBUG] setup: 无LoRA subject摆位 delta={POS_DELTA} steps={STEPS} "
      f"cond={COND_SIZE}px target={TARGET_SIZE}px out={OUT_DIR}")

16:46:18 [INFO] cudnn disabled (workaround for VAE CUDNN_STATUS_NOT_INITIALIZED)
16:46:20 [WARNING] Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.8.0+cu128).
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`


[DEBUG] setup: 无LoRA subject摆位 delta=(0, 32) steps=28 cond=512px target=512px out=repro/subject_base_compare_512


In [2]:
# 强制走本地 HF cache + 直接用 snapshot 路径(双保险,完全绕开网络)
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

LOCAL_FLUX_PATH = "/home/wuwenxuan03/.cache/huggingface/hub/models--black-forest-labs--FLUX.1-dev/snapshots/3de623fc3c33e44ffbe2bad470d0f45bccf2eb21"
print("LOCAL_FLUX_PATH exists:", os.path.isdir(LOCAL_FLUX_PATH))

LOCAL_FLUX_PATH exists: True


In [3]:
# 加载 pipeline(dispatch=auto → 24G 卡走 2gpu 拆分)。不调 attach_lora —— 纯基座,
# adapter=None 时 lora_forward 直接走原始权重(flux_omini.py 的 None 分支)。
pipe = load_pipeline(LOCAL_FLUX_PATH, dispatch="auto", gpu0=2,gpu1=3)

16:46:27 [INFO] GPU 0: NVIDIA GeForce RTX 4090 | 23.6 GB
16:46:27 [INFO] GPU 1: NVIDIA GeForce RTX 4090 | 23.6 GB
16:46:27 [INFO] GPU 2: NVIDIA GeForce RTX 4090 | 23.6 GB
16:46:27 [INFO] GPU 3: NVIDIA GeForce RTX 4090 | 23.6 GB
16:46:27 [INFO] GPU 4: NVIDIA GeForce RTX 4090 | 23.6 GB
16:46:27 [INFO] GPU 5: NVIDIA GeForce RTX 4090 | 23.6 GB
16:46:27 [INFO] GPU 6: NVIDIA GeForce RTX 4090 | 23.6 GB
16:46:27 [INFO] GPU 7: NVIDIA GeForce RTX 4090 | 23.6 GB
16:46:27 [INFO] dispatch=auto -> 选择 2gpu(GPU0 24 GB)
16:46:27 [INFO] 加载 FLUX pipeline: /home/wuwenxuan03/.cache/huggingface/hub/models--black-forest-labs--FLUX.1-dev/snapshots/3de623fc3c33e44ffbe2bad470d0f45bccf2eb21 (bf16, 2gpu 手工 dispatch)


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers
16:46:38 [INFO] dispatch 完成: cuda:2 = encoders+vae+latents(主场), cuda:3 = transformer
16:46:38 [INFO] 已安装 tx bridge:输入 -> transformer 卡,输出 -> 主场卡


In [4]:
# 6 张主体条件图。前 5 条 prompt 逐字取自官方 examples/subject*.ipynb;
# book.jpg 官方只用于 inpainting,此条按官方 "this item" 句式自拟。
CASES = [
    ("penguin", "assets/penguin.jpg",
     "On Christmas evening, on a crowded sidewalk, this item sits on the road, "
     "covered in snow and wearing a Christmas hat."),
    ("rc_car", "assets/rc_car.jpg",
     "A film style shot. On the moon, this item drives across the moon surface. "
     "The background is that Earth looms large in the foreground."),
    ("tshirt", "assets/tshirt.jpg",
     "On the beach, a lady sits under a beach umbrella. She's wearing this shirt and has "
     "a big smile on her face, with her surfboard hehind her. The sun is setting in the "
     "background. The sky is a beautiful shade of orange and purple."),
    ("clock", "assets/clock.jpg",
     "In a Bauhaus style room, this item is placed on a shiny glass table, with a vase of "
     "flowers next to it. In the afternoon sun, the shadows of the blinds are cast on the wall."),
    ("oranges", "assets/oranges.jpg",
     "A very close up view of this item. It is placed on a wooden table. The background is "
     "a dark room, the TV is on, and the screen is showing a cooking show."),
    ("book", "assets/book.jpg",  # prompt 自拟(官方无 subject 用例)
     "This item is placed on a wooden desk in a library, next to a cup of coffee, "
     "under the warm light of a reading lamp."),
]

def build_condition(image_path):
    # 官方预处理:center-crop 成正方形再 resize(pipeline 不会自动做)
    img = Image.open(image_path).convert("RGB")
    w, h = img.size; m = min(w, h)
    img = img.crop(((w - m) // 2, (h - m) // 2, (w + m) // 2, (h + m) // 2))
    img = img.resize((COND_SIZE, COND_SIZE))
    # 主体驱动:原图直接作条件(不做 canny),错开摆位;adapter=None → 纯基座
    return img, Condition(img, None, position_delta=POS_DELTA)

def run_one(prompt, condition, seed, kv_cache):
    # WHY 每次重建 generator:两种模式必须从同一噪声出发,差异才全部归因于注意力模式
    g = torch.Generator(device="cuda:0").manual_seed(seed)
    t0 = time.perf_counter()
    out = generate(pipe, prompt=prompt, conditions=[condition],
                   height=TARGET_SIZE, width=TARGET_SIZE,
                   num_inference_steps=STEPS, guidance_scale=3.5,
                   generator=g, kv_cache=kv_cache)
    return out.images[0], time.perf_counter() - t0

In [5]:
# 主循环:6 case × 2 seed × 2 模式 = 24 次生成
records = []
for name, path, prompt in CASES:
    cond_img, condition = build_condition(path)
    cond_img.save(f"{OUT_DIR}/{name}_cond.png")
    for seed in SEEDS:
        row = {"case": name, "seed": seed, "prompt": prompt}
        for mode, kv in (("default", False), ("kvcache", True)):
            img, dt = run_one(prompt, condition, seed, kv)
            fp = f"{OUT_DIR}/{name}_s{seed}_{mode}.png"
            img.save(fp)
            row[mode] = fp; row[f"{mode}_sec"] = round(dt, 1)
        records.append(row)
        print(f"[DEBUG] run_one: {name} seed={seed} "
              f"default={row['default_sec']}s kvcache={row['kvcache_sec']}s "
              f"speedup={row['default_sec']/row['kvcache_sec']:.2f}x")

with open(f"{OUT_DIR}/records.json", "w") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)
print(f"完成 {len(records)} 组对照,已写 {OUT_DIR}/records.json")

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

[DEBUG] run_one: penguin seed=42 default=10.3s kvcache=6.1s speedup=1.69x


  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

...

  0%|          | 0/28 [00:00<?, ?it/s]

[DEBUG] run_one: book seed=42 default=9.9s kvcache=6.2s speedup=1.60x


  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

[DEBUG] run_one: book seed=1234 default=9.9s kvcache=6.2s speedup=1.60x
完成 12 组对照,已写 repro/subject_base_compare_512/records.json


In [6]:
# 可视化:复用独立脚本,产物 $OUT_DIR/comparison_grid.png
# (脚本按 records.json 渲染 [prompt | 条件图 | 默认 | KV-Cache] 网格,行数自适应 6 case)
!python repro/visualize_quality_compare.py \
    --records $OUT_DIR/records.json \
    --out-dir $OUT_DIR

[INFO] repo_root  : /home/wuwenxuan03/OminiControl
[INFO] records    : repro/subject_base_compare_512/records.json
[INFO] out_dir    : repro/subject_base_compare_512
[INFO] 已生成     : repro/subject_base_compare_512/comparison_grid.png  (5641 KB)

对比摘要  (12 组, 同 LoRA 同 seed)
  penguin  s  42  default= 10.3s  kvcache=  6.1s  speedup=1.69x
  penguin  s1234  default=  9.8s  kvcache=  6.2s  speedup=1.58x
  rc_car   s  42  default=  9.8s  kvcache=  6.2s  speedup=1.58x
  rc_car   s1234  default=  9.8s  kvcache=  6.2s  speedup=1.58x
  tshirt   s  42  default=  9.9s  kvcache=  6.2s  speedup=1.60x
  tshirt   s1234  default=  9.9s  kvcache=  6.2s  speedup=1.60x
  clock    s  42  default=  9.9s  kvcache=  6.2s  speedup=1.60x
  clock    s1234  default=  9.9s  kvcache=  6.2s  speedup=1.60x
  oranges  s  42  default=  9.9s  kvcache=  6.3s  speedup=1.57x
  oranges  s1234  default=  9.9s  kvcache=  6.2s  speedup=1.60x
  book     s  42  default=  9.9s  kvcache=  6.2s  speedup=1.60x
  book     s1234  defa

In [ ]:
# GSB 标注骨架(格式对齐 scripts/compute_gsb.py),盲评随机换边
rng = random.Random(0)
annotations, side_map = {}, {}
for r in records:
    cid = f"{r['case']}_s{r['seed']}"
    flip = rng.random() < 0.5
    side_map[cid] = {"A": "kvcache" if flip else "default",
                     "B": "default" if flip else "kvcache"}
    annotations[cid] = {"prompt": r["prompt"], "choices": {"quality": ""}}

with open(f"{OUT_DIR}/annotations.json", "w") as f:
    json.dump({"annotations": annotations}, f, ensure_ascii=False, indent=2)
with open(f"{OUT_DIR}/side_map.json", "w") as f:
    json.dump(side_map, f, ensure_ascii=False, indent=2)
print(f"标注后统计: python scripts/compute_gsb.py {OUT_DIR}/annotations.json")

## 备注

- 主体驱动摆位下条件 token 数 = 主图 token 数(条件尺寸=目标尺寸),任何 `TARGET_SIZE`
  下条件都占序列一半 —— KV-Cache 的理论收益上限恒定(≈2 倍计算量差),
  与 stage5 的"占比随分辨率稀释"不同,speedup 数据可以直接验证这一点。
- 后续要挂官方 subject LoRA 复跑时:`Condition(img, "subject", position_delta=(0, 32))`,
  dev 上加 `image_guidance_scale=1.5`,即可复用本 notebook 其余全部代码。
- 与 stage5 同 seed(42/1234)同 steps,可跨实验横向比对两种摆位下的机制差距。